# 03 — Post-Clustering Analysis & CVI
## MIT Sloan SSAC 2026 Hackathon

**Mission:** Turn cluster assignments into business insights, compute the
Commercial Value Index, and generate every chart needed for the presentation.

This is where we go from "we found 4 groups" to "here's why this one group
is worth $200M and nobody's talking to them." The analysis that makes
sponsors reach for their checkbooks.

---

## 1. Setup + Load Clustered Data

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../scripts'))
from preprocessing import *
from clustering import *
from profiling import *
from visualization import *

%matplotlib inline

# Load clustered data from step 02
df = pd.read_csv('../data/clustered_data.csv')
labels = df['cluster'].values
col_types = identify_column_types(df.drop(columns=['cluster']))

k = len(df['cluster'].unique())
print(f'✅ Loaded {len(df):,} rows, {k} clusters')
print(f'   Cluster sizes: {df["cluster"].value_counts().to_dict()}')

In [ ]:
# ======================================
# SET YOUR CLUSTER NAMES HERE
# ======================================
# Copy from 02_clustering or set new ones:

cluster_names = {i: f'Segment {i+1}' for i in range(k)}

# MANUAL OVERRIDE — fill these in once you know your data:
# cluster_names = {
#     0: 'The Values Advocate',
#     1: 'The Silent Superfan',
#     2: 'The Social Spectator',
#     3: 'The Casual Curious',
# }

print(f'Cluster names: {cluster_names}')

## 2. Commercial Value Index (CVI)

⚠️ **THIS IS THE HACKATHON-DAY MONEY CELL.**

Map your actual dataset columns to each CVI sub-score below.
This is where domain knowledge meets data — you need to decide
which columns represent spending, brand receptivity, etc.

**CVI Formula:**
```
CVI = 0.30(Spending) + 0.25(Brand Receptivity) + 0.20(Engagement)
    + 0.15(Social Amplification) + 0.10(Growth Potential)
```

In [ ]:
# ============================================================
# ⬆️⬆️⬆️  MAP YOUR ACTUAL COLUMNS HERE  ⬆️⬆️⬆️
# ============================================================
# For each sub-score, list the column names from YOUR dataset
# that correspond to that concept. Leave empty if no match.
#
# The function averages the listed columns per sub-score,
# normalizes each to 0-1, then applies the weights.
# ============================================================

cvi_mapping = {
    'spending_score': [],       # e.g., ['ticket_spend', 'merch_spend', 'food_bev_spend']
    'brand_receptivity': [],    # e.g., ['sponsor_awareness', 'purchase_intent', 'brand_recall']
    'engagement_depth': [],     # e.g., ['games_attended', 'app_sessions', 'content_consumed']
    'social_amplification': [], # e.g., ['social_shares', 'social_follows', 'ugc_posts']
    'growth_potential': [],     # e.g., ['age_inverted', 'tenure_inverted', 'avidity_trend']
}

# NOTE: For 'growth_potential', you might want to invert age and tenure:
# df['age_inverted'] = df['age'].max() - df['age']  # younger = higher potential
# df['tenure_inverted'] = df['fan_tenure'].max() - df['fan_tenure']  # newer = higher potential

df = compute_cvi(df, labels, col_types, mapping=cvi_mapping)

## 3. Segment Size vs. Value Analysis

**This is Slide 7 material** — the chart that shows which segments
punch above their weight in commercial value.

Above the diagonal = goldmine. Below = growth opportunity.

In [ ]:
sv = segment_size_and_value(df, labels)
display(sv)

fig = plot_segment_size_value(sv, cluster_names, save_path='../outputs/size_vs_value.png')
fig.show()

## 4. Cluster Comparison Index

The heatmap that reveals at a glance what makes each segment unique.
Red = over-indexes (>120), Blue = under-indexes (<80).

In [ ]:
comparison = generate_cluster_comparison(df, labels, col_types)
display(comparison)

fig_heatmap = plot_cluster_heatmap(comparison, cluster_names, save_path='../outputs/cluster_heatmap.png')
plt.show()

## 5. Radar Chart

The "personality wheel" for each persona. Pick 6-8 features that
best differentiate the clusters. More features = messier chart.

In [ ]:
# Auto-select most differentiating features from the comparison index
if len(comparison.columns) > 0:
    # Pick features with highest variance across clusters (most differentiating)
    feature_variance = comparison.var().sort_values(ascending=False)
    radar_features = feature_variance.head(8).index.tolist()
    print(f'Top differentiating features: {radar_features}')
    
    # Need to get profiles for radar
    profiles = profile_clusters(df, labels, col_types)
    fig_radar = plot_radar(profiles, radar_features, cluster_names, save_path='../outputs/radar.png')
    fig_radar.show()
else:
    print('No comparison data available for radar chart.')

In [ ]:
# ======================================
# MANUAL OVERRIDE — pick your own radar features
# ======================================
# radar_features = ['spending', 'engagement', 'social', 'brand_recall', 'age', 'loyalty']
# fig_radar = plot_radar(profiles, radar_features, cluster_names, save_path='../outputs/radar.png')
# fig_radar.show()

## 6. CVI Bar Chart

In [ ]:
fig_cvi = plot_cvi_bar(sv, cluster_names, save_path='../outputs/cvi_bar.png')
fig_cvi.show()

## 7. Deep Dive: The Undervalued Segment

Find the segment that over-indexes on commercial value relative to
its size. This is your "hidden gem" story — the one nobody's marketing to.

In [ ]:
# Find the most "undervalued" segment (highest value efficiency)
if 'value_efficiency' in sv.columns:
    star_segment = sv.loc[sv['value_efficiency'].idxmax()]
    star_id = int(star_segment['cluster'])
    star_name = cluster_names.get(star_id, f'Cluster {star_id}')
    
    print(f'⭐ Most value-efficient segment: {star_name}')
    print(f'   {star_segment["pct_of_total"]}% of fans, '
          f'{star_segment["pct_of_total_cvi"]}% of value')
    print(f'   Value efficiency: {star_segment["value_efficiency"]:.2f}x')
    print()
    
    # Profile deep-dive
    star_fans = df[df['cluster'] == star_id]
    print(f'Profile of {star_name} ({len(star_fans):,} fans):')
    display(star_fans.describe().round(2))
else:
    print('Run CVI computation first (cell 2).')

In [ ]:
# Feature distribution deep-dives for the star segment
# Uncomment and change the feature name:
#
# feature_to_explore = 'your_column_name'  # CHANGE THIS
# fig = plot_feature_distributions(df, feature_to_explore, labels, cluster_names,
#                                  save_path=f'../outputs/dist_{feature_to_explore}.png')
# plt.show()

## 8. Deep Dive: Iso-Fan / Silent Superfan Analysis

The "iso-fan" — high spending, low social engagement, low community
participation. They love the sport but nobody knows they exist.
They're the introverts of fandom. The dark matter of the fan universe.

Brands miss them because they're invisible on social media.

In [ ]:
# ======================================
# ISO-FAN DETECTION — adapt columns to your data
# ======================================
# Uncomment and fill in with your actual column names:
#
# spend_col = 'total_spend'    # CHANGE to your spending column
# social_col = 'social_score'  # CHANGE to your social engagement column
#
# # Define iso-fan criteria: high spend (top quartile), low social (bottom quartile)
# spend_threshold = df[spend_col].quantile(0.75)
# social_threshold = df[social_col].quantile(0.25)
#
# iso_fans = df[(df[spend_col] >= spend_threshold) & (df[social_col] <= social_threshold)]
#
# print(f'🔍 Found {len(iso_fans):,} iso-fans ({len(iso_fans)/len(df)*100:.1f}% of total)')
# print(f'   Criteria: {spend_col} >= {spend_threshold:.2f} AND {social_col} <= {social_threshold:.2f}')
# print(f'\nIso-fan cluster distribution:')
# print(iso_fans['cluster'].value_counts())
#
# # Compare iso-fans vs population
# print(f'\nIso-fan profile vs population:')
# comparison_cols = [c for c in df.select_dtypes(include=np.number).columns if c != 'cluster']
# pop_means = df[comparison_cols].mean()
# iso_means = iso_fans[comparison_cols].mean()
# diff = ((iso_means / pop_means - 1) * 100).sort_values(ascending=False)
# print(diff.round(1))

## 9. Export All Visualizations

In [ ]:
# Save everything to outputs/
export_all_charts('../outputs/')

# Also save the final clustered data with CVI
df.to_csv('../data/clustered_data_with_cvi.csv', index=False)
print(f'\n✅ Final data saved: data/clustered_data_with_cvi.csv')
print(f'   Ready for Streamlit dashboard: streamlit run app/dashboard.py')

## 10. Presentation-Ready Findings Summary

Fill this in as you go. This is your presentation script.

---

### Finding 1: [Title]
**The insight:** 

**The evidence:** 

**The activation opportunity:** 

**The dollar figure:** 

---

### Finding 2: [Title]
**The insight:** 

**The evidence:** 

**The activation opportunity:** 

**The dollar figure:** 

---

### Finding 3: [Title]
**The insight:** 

**The evidence:** 

**The activation opportunity:** 

**The dollar figure:** 

---

*Remember: Every finding needs a "so what" and a "now what."*
*The judges don't care about your R² score. They care about revenue.*